# Formateo de nombres dataset - quechua

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, re, shutil, json
from pathlib import Path
import pandas as pd
from datetime import datetime

In [ ]:
BASE_DIR = Path('/content/drive/MyDrive/TP1/modelo2/Quechua_Collao_Corpus')
AUDIOS_DIR = BASE_DIR / 'Audios'
LABELS_CSV = BASE_DIR / 'Labels' / 'Labels' / 'Labels.csv'

In [ ]:
NDEC = 2 #numero de decimales

In [ ]:
BACKUP_DIR = BASE_DIR / '_rename_backup' # backup de nuevos nombres con nombres originales
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
MAP_PATH = BACKUP_DIR / f"rename_map_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

In [ ]:
assert AUDIOS_DIR.is_dir(), f"No existe la carpeta de audios: {AUDIOS_DIR}"
assert LABELS_CSV.is_file(), f"No se encontró Labels.csv en: {LABELS_CSV}"

df = pd.read_csv(LABELS_CSV)

confirmamos valores continuos VAD

In [ ]:
df['Valence'].unique()

array(['1.25', '3.5', '3.25', '3.75', '2', '3', '2.75', '2.25', '4.25',
       '1', '1.5', '1.75', '4.75', '4.5', '4', '2.5', '5',
       '2.333.333.333'], dtype=object)

In [ ]:
df['Arousal'].unique()

array([4.75, 2.  , 2.75, 2.25, 3.5 , 1.  , 1.5 , 2.5 , 4.25, 4.5 , 4.  ,
       5.  , 1.75, 1.25, 3.75, 3.  , 3.25])

In [ ]:
df['Dominance'].unique()

array([4.5 , 2.25, 2.5 , 3.75, 3.67, 4.75, 2.75, 1.5 , 3.  , 4.  , 4.25,
       1.25, 3.5 , 1.75, 3.25, 5.  , 2.  , 1.  ])

Verificamos faltantes

In [ ]:
df.isna().any()

,0
Audio,False
Valence,False
Arousal,False
Dominance,False


mapeo de valores VAD de cada audio

In [ ]:
def clean_first_float(s: pd.Series) -> pd.Series:
    return (s.astype(str)
             .str.replace(',', '.', regex=False)                 # decimales con punto
             .str.replace(r'\s+', '', regex=True)                # sin espacios
             .str.extract(r'(-?\d+(?:\.\d+)?)', expand=False)    # primer número válido
             .astype(float))

for col in ['Valence','Arousal','Dominance']:
    df[col] = clean_first_float(df[col])


In [ ]:
for col in ['Valence', 'Arousal', 'Dominance']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
df['Audio'] = df['Audio'].astype(str).str.strip()
labels_map = {
    k: (float(v), float(a), float(d))
    for k, v, a, d in df[['Audio','Valence','Arousal','Dominance']].itertuples(index=False, name=None)
}

In [ ]:
num_pat = re.compile(r'(\d+)')

extraccion de los ids

In [ ]:
def extract_id(stem: str) -> str:
    """
    Extrae el primer bloque de dígitos del nombre (sin extensión).
    Ej: '10001', '10001_take2' -> '10001'
    """
    m = num_pat.search(stem)
    return m.group(1) if m else None

def fmt(x: float) -> str:
    # formato con NDEC decimales, quita ceros y punto final si no hace falta
    s = f"{x:.{NDEC}f}"
    s = s.rstrip('0').rstrip('.') if '.' in s else s
    return s

def build_suffix(v, a, d):
    return f"-{fmt(v)}-{fmt(a)}-{fmt(d)}"

arrays temporales

In [ ]:
renames = []
not_found = []
already_ok = []
collisions = []

nuevos nombres y su guardado en ruta

In [ ]:
for wav in sorted(AUDIOS_DIR.glob("*.wav")):
    stem = wav.stem  # sin extensión
    aid = extract_id(stem)
    if not aid:
        # si no hay ID
        continue

    if aid not in labels_map:
        not_found.append(wav.name)
        print("no se encontro:c")
        continue

    v, a, d = labels_map[aid]
    suffix = build_suffix(v, a, d)

    # Evita duplicar si ya tiene el sufijo correcto
    if stem.endswith(suffix):
        already_ok.append(wav.name)
        continue

    new_name = f"{stem}{suffix}{wav.suffix}"
    new_path = wav.with_name(new_name)

    if new_path.exists():
        collisions.append((wav.name, new_name))
        continue

    renames.append((wav, new_path))

In [ ]:
print(f"Audios totales encontrados: {len(list(AUDIOS_DIR.glob('*.wav')))}")
print(f"Con ID no mapeado (no están en CSV): {len(not_found)}")
print(f"Ya tenían el sufijo correcto: {len(already_ok)}")
print(f"Propuestos para renombrar: {len(renames)}")
print(f"Choques de nombre potenciales: {len(collisions)}")

Audios totales encontrados: 12420
Con ID no mapeado (no están en CSV): 0
Ya tenían el sufijo correcto: 0
Propuestos para renombrar: 12420
Choques de nombre potenciales: 0


In [ ]:
for old, new in renames[:10]:
    print("  ", old.name, "->", new.name)

   10001.wav -> 10001-1.25-4.75-4.5.wav
   10002.wav -> 10002-3.5-2-2.25.wav
   10003.wav -> 10003-3.25-2.75-2.5.wav
   10004.wav -> 10004-3.75-2.25-2.5.wav
   10005.wav -> 10005-3.5-3.5-3.75.wav
   10006.wav -> 10006-3.75-2.25-3.67.wav
   10007.wav -> 10007-2-4.75-4.75.wav
   10008.wav -> 10008-3.5-2.75-2.75.wav
   10009.wav -> 10009-3.75-1-1.5.wav
   10010.wav -> 10010-3.75-1.5-1.5.wav


In [ ]:
renames[0]

(PosixPath('/content/drive/MyDrive/TP1/modelo2/Quechua_Collao_Corpus/Audios/10001.wav'),
 PosixPath('/content/drive/MyDrive/TP1/modelo2/Quechua_Collao_Corpus/Audios/10001-1.25-4.75-4.5.wav'))

In [ ]:
if renames:
    pd.DataFrame(
        [(old.name, new.name) for old, new in renames],
        columns=['old_name', 'new_name']
    ).to_csv(MAP_PATH, index=False)
    print(f"\nMapa de renombrado guardado en: {MAP_PATH}")


Mapa de renombrado guardado en: /content/drive/MyDrive/TP1/modelo2/Quechua_Collao_Corpus/_rename_backup/rename_map_20250919_215125.csv


In [ ]:
applied = 0
for old, new in renames:
    old.rename(new)
    applied += 1
print(f"\nRenombrados aplicados: {applied}")


Renombrados aplicados: 12420


verificación de dominancia (data desbalanceada anteriormente)

In [ ]:
vad_re2 = re.compile(r'-(?P<V>-?\d+(?:\.\d+)?)-(?P<A>-?\d+(?:\.\d+)?)-(?P<D>-?\d+(?:\.\d+)?)\.wav$', re.I)

rows = []
for p in sorted(AUDIOS_DIR.rglob('*.wav')):
    m = vad_re2.search(p.name)
    if not m:
        continue
    v = float(m.group('V')); a = float(m.group('A')); d = float(m.group('D'))
    # ID = primer bloque numérico del nombre
    clip_id = p.stem.split('-')[0]
    rows.append(dict(path=str(p), file=p.name, clip_id=clip_id, V=v, A=a, D=d))

df2 = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError("No se encontraron .wav con sufijo -V-A-D. Revisa las rutas.")

# Detecta escala de etiquetas para fijar el umbral equivalente
# (Si están en 0..1, 'D<3' equivale a D<0.5)
thr = 2.4
if df2['D'].max() <= 1.2:  # probablemente ya escaladas a 0..1
    thr = 0.5

sel = df2[df2['D'] < thr].copy()

si hay buena cantidad de dominancia menor a el intermedio

In [ ]:
sel

,path,file,clip_id,V,A,D
1,/content/drive/MyDrive/TP1/modelo2/Quechua_Col...,10002-3.5-2-2.25.wav,10002,3.50,2.00,2.25
8,/content/drive/MyDrive/TP1/modelo2/Quechua_Col...,10009-3.75-1-1.5.wav,10009,3.75,1.00,1.50
9,/content/drive/MyDrive/TP1/modelo2/Quechua_Col...,10010-3.75-1.5-1.5.wav,10010,3.75,1.50,1.50
19,/content/drive/MyDrive/TP1/modelo2/Quechua_Col...,10020-3.25-1.5-2.25.wav,10020,3.25,1.50,2.25
27,/content/drive/MyDrive/TP1/modelo2/Quechua_Col...,10028-3.5-1.5-1.25.wav,10028,3.50,1.50,1.25
...,...,...,...,...,...,...
12395,/content/drive/MyDrive/TP1/modelo2/Quechua_Col...,22396-2-4-2.25.wav,22396,2.00,4.00,2.25
12396,/content/drive/MyDrive/TP1/modelo2/Quechua_Col...,22397-3.5-2.75-2.25.wav,22397,3.50,2.75,2.25
12398,/content/drive/MyDrive/TP1/modelo2/Quechua_Col...,22399-3-3-1.75.wav,22399,3.00,3.00,1.75
12402,/content/drive/MyDrive/TP1/modelo2/Quechua_Col...,22403-2-2.5-2.wav,22403,2.00,2.50,2.00


## funciones

In [ ]:
def is_under(path: Path, root: Path) -> bool:
    try:
        return path.resolve().is_relative_to(root.resolve())
    except AttributeError:
        try:
            path.resolve().relative_to(root.resolve())
            return True
        except Exception:
            return False

In [ ]:
def safe_speaker_id(path: Path) -> str:
    """
    - GlobalV1: usa la carpeta inmediatamente debajo de GLOBALV1_DIR (ej. 'audio1')
    - Quechua: usa el prefijo numérico al inicio del archivo (ej. '10002' en '10002-3.5-2-2.25.wav')
    - Si no encuentra nada, devuelve cadena vacía.
    """
    p = path.resolve()

    # Caso GlobalV1
    if is_under(p, GLOBALV1_DIR):
        rel = p.relative_to(GLOBALV1_DIR.resolve())
        if len(rel.parts) >= 2:
            return rel.parts[0]  # carpeta del speaker
        # Fallback: usa el nombre de la carpeta contenedora
        if p.parent != GLOBALV1_DIR:
            print("alerta - nombre de carpeta ")
            return p.parent.name

    # Caso Quechua (prefijo numérico en el nombre)
    m = re.match(r"^(\d+)", p.stem)
    if m:
        return m.group(1)

    return ""

In [ ]:
def parse_labels_quechua(stem: str) -> Optional[Tuple[float, float, float]]:
    """
    Formato:  <id>-<v>-<a>-<d>, floats (p.ej. 1.25)
    """
    parts = stem.split("-")
    if len(parts) != 4:
        return None
    try:
        val = float(parts[1])
        aro = float(parts[2])
        dom = float(parts[3])
        return val, aro, dom
    except Exception:
        return None

In [ ]:
def parse_labels_global(stem: str) -> Optional[Tuple[float, float, float]]:
    """
    GlobalV1: nombres terminan en -V-A-D (enteros 01..05).
    Acepta: 'audio1_1-02-02-03', etc.
    objetivo: buscar al final ...-d1-d2-d3
    """
    m = re.search(r"-([0-9]+)-([0-9]+)-([0-9]+)$", stem)
    if not m:
        return None
    try:
        va = float(int(m.group(1)))  # '02' -> 2.0
        ar = float(int(m.group(2)))
        do = float(int(m.group(3)))
        return va, ar, do
    except Exception:
        return None

In [ ]:
def clip_1_5(x: float) -> float:
    return float(np.clip(x, 1.0, 5.0))

In [ ]:
def make_row(path: Path, corpus: str, labels: Tuple[float, float, float]) -> Dict:
    va, ar, do = labels
    return dict(
        path=str(path.resolve()),
        corpus=corpus,
        lang=LANG_BY_CORPUS.get(corpus, ""),
        speaker_id=safe_speaker_id(path),
        valence=clip_1_5(va),
        arousal=clip_1_5(ar),
        dominance=clip_1_5(do),
        clip_id=path.stem  # id único por archivo
    )

In [ ]:
def scan_corpus(root: Path, corpus_name: str) -> Tuple[List[Dict], List[Tuple[str, str]]]:
    rows: List[Dict] = []
    skipped: List[Tuple[str, str]] = []

    for p in root.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix not in ALLOWED_EXT:
            continue

        stem = p.stem

        if corpus_name == "quechua":
            labels = parse_labels_quechua(stem)
        else:  # globalv1
            labels = parse_labels_global(stem)

        if labels is None:
            skipped.append((str(p), "no_labels"))
            continue

        # sanity check 1..5
        if not all(1.0 <= v <= 5.0 for v in labels):
            skipped.append((str(p), f"out_of_range:{labels}"))
            continue

        rows.append(make_row(p, corpus_name, labels))

    return rows, skipped

# Información de ambos datasets en csv

In [ ]:
import os, re, math, json
from pathlib import Path
from typing import Optional, Tuple, List, Dict
import numpy as np
import pandas as pd

In [ ]:
# RUTAS
QUECHUA_DIR        = Path("/content/drive/MyDrive/TP1/data/Quechua_Collao_Corpus/Audios")
GLOBALV1_DIR       = Path("/content/drive/MyDrive/TP1/data/Corpus_Globalv1")
OUT_UNIFIED       = Path("/content/drive/MyDrive/TP1/data/unificado.csv")

In [ ]:
LANG_BY_CORPUS = {
    "quechua": "qu",
    "globalv1": "es",
}

In [ ]:
ALLOWED_EXT = {".wav", ".WAV"}

In [ ]:
all_rows: List[Dict] = []
all_skipped: List[Tuple[str, str]] = []

# Quechua (archivos en raíz)
if QUECHUA_DIR.exists():
    rows_q, skip_q = scan_corpus(QUECHUA_DIR, "quechua")
    all_rows.extend(rows_q); all_skipped.extend(skip_q)
    print(f"[quechua] válidos: {len(rows_q)} | saltados: {len(skip_q)}")
else:
    print(f"[quechua] Ruta no existe: {QUECHUA_DIR}")

# GlobalV1 (carpetas con audios)
if GLOBALV1_DIR.exists():
    rows_g, skip_g = scan_corpus(GLOBALV1_DIR, "globalv1")
    all_rows.extend(rows_g); all_skipped.extend(skip_g)
    print(f"[globalv1] válidos: {len(rows_g)} | saltados: {len(skip_g)}")
else:
    print(f"[globalv1] Ruta no existe: {GLOBALV1_DIR}")

if not all_rows:
    print("No se recolectaron filas. Revisa rutas y patrones de nombres.")

[quechua] válidos: 12420 | saltados: 0
[globalv1] válidos: 3763 | saltados: 1


In [ ]:
df = pd.DataFrame(all_rows)
# Ordena columnas para legibilidad
cols = ["path", "corpus", "lang", "speaker_id", "clip_id", "valence", "arousal", "dominance"]
df = df[cols]

OUT_UNIFIED.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_UNIFIED, index=False, encoding="utf-8")
print(f"\nGuardado unified: {OUT_UNIFIED}")
print(df.head(8))

# Resumen útil
print("\n=== Resumen ===")
print(df.groupby(["corpus", "lang"]).size().rename("n").reset_index())
for col in ["valence", "arousal", "dominance"]:
    desc = df[col].describe()
    print(f"\n{col}:\n{desc}")

if all_skipped:
    # Muestra los 10 primeros saltos por diagnóstico
    print("\nEjemplos saltados (máx 10):")
    for path, why in all_skipped[:10]:
        print(f"- {why}: {path}")



Guardado unified: /content/drive/MyDrive/TP1/data/unificado.csv
                                                path   corpus lang speaker_id  \
0  /content/drive/.shortcut-targets-by-id/1Kj75Ss...  quechua   qu      21421   
1  /content/drive/.shortcut-targets-by-id/1Kj75Ss...  quechua   qu      21422   
2  /content/drive/.shortcut-targets-by-id/1Kj75Ss...  quechua   qu      21423   
3  /content/drive/.shortcut-targets-by-id/1Kj75Ss...  quechua   qu      21424   
4  /content/drive/.shortcut-targets-by-id/1Kj75Ss...  quechua   qu      21425   
5  /content/drive/.shortcut-targets-by-id/1Kj75Ss...  quechua   qu      21426   
6  /content/drive/.shortcut-targets-by-id/1Kj75Ss...  quechua   qu      21427   
7  /content/drive/.shortcut-targets-by-id/1Kj75Ss...  quechua   qu      21428   

                clip_id  valence  arousal  dominance  
0   21421-3.25-2.5-2.75     3.25     2.50       2.75  
1      21422-1.75-3.5-4     1.75     3.50       4.00  
2   21423-2.75-2.5-2.75     2.75     2.5